# **Spring 2026 CSE 538 - Natural Language Processing Assignment 3**
---
**Due date: May 6th**

**Disclaimer/License**: This code is only for school assignment purpose, and **any version of this should NOT be shared publicly on github or otherwise even after semester ends**.
Public availability of answers devalues usability of the assignment and work of several TAs who have contributed to this.
We hope you'll respect this restriction.

**Overview**
--
This assignment is a simple introduction to prompting and the potential for artifact-based reasoning in reward modeling.

***Part I: Prompting for Sentiment Classification***
In the first part, you will test out the basics of prompting with Large Language Models (LLMs) on the sentiment classification task, we have been doing in the previous assignments.


The script for prompting and analyzing LLMs for sentiment classification involves the following steps. We have implemented most of the steps. You will be asked to implement the steps marked TODO.

Steps:

  - walk through example prompt, execute the commands and understand the flow
  - design different prompts for the sentiment classification task
  - calculate accuracy of using each prompt for this task
  - characterize model behaviour by analyzing different attributes of the prompt

**NOTE**: Please only make your code edits in the TODO(students) blocks in the notebook and make sure you have run the previous notebook cells before running the latter ones. Add comments to explain your code well and make sure to use relevant identifier names.

**NOTE**: You can use the GPU runtime in this assignment through Google Colab freely available resources. However, use CPU in your development phase. Once your implementation is complete, switch to GPU and run the experiments.

**Tips**: To load a 1B model such as *Llama-3.2-1B-Instruct* with half precision, 1 x 2 = 2 GB of GPU RAM is needed, to run inference ~ 3-4 GB is required. Therefore, you should be able to run this on your local systems, in the rare event that Google Colab servers are not available.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# you can create a NLP assignment3 folder in your Google drive upload the data folder there
base_dir = "YOUR GOOGLE DRIVE PATH"

In [ ]:
%cd $base_dir

In [ ]:
!pip install datasets --upgrade
!pip install tqdm

In [ ]:
import pandas as pd
import requests
from datasets import load_dataset
from tqdm import tqdm

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# Install necessary libraries
!pip install transformers accelerate bitsandbytes

In [ ]:
data = load_dataset("imdb")

In [ ]:
def extract_input_outputs(dataset):
    """
    input: dataset -> List[Dict]
    Each dict contains two keys - text and label.
    text contains the text for which sentiment has to be labeled.
    label is an integer denoting the gold sentiment.
    output: inputs -> List, outputs -> List
    inputs is a list of texts from dataset
    outputs is a list of labels from dataset
    """

    inputs, outputs = [], []
    # TODO(students): start
    # Collect the review text and gold label from each dataset example.
    for example in dataset:
        inputs.append(example["text"])
        outputs.append(example["label"])
    # TODO(students): end
    return inputs, outputs

In [ ]:
def create_prompt_inputs(input_texts, prompt_str):
    """
    input: input_texts -> List, prompt_str -> str
    input_texts is an array of reviews from the imdb dataset
    prompt_str is a template prompt with a placeholder to be filled in using a review
    Together, the prompt_str template and a review is used to create an input for the LLM
    output: prompt_inputs -> List
    prompt_inputs is an array of strings.
    Each element is the prompt template prompt_str with the placeholder filled by the review.
    """
    prompt_inputs = []
    # TODO(students): start
    # Fill the prompt template once for every review in the batch.
    for input_text in input_texts:
        prompt_inputs.append(prompt_str.format(input_text=input_text))
    # TODO(students): end
    return prompt_inputs

In [ ]:
def calculate_accuracy(transformed_model_outputs, ground_truth_labels):
    total, correct = 0, 0
    for predicted_label, ground_truth in zip(transformed_model_outputs, ground_truth_labels):
        if predicted_label == ground_truth:
            correct += 1
        total += 1
    accuracy = round(correct * 100 / total, 2)
    return accuracy

In [ ]:
def query(model_inputs):
    # Utility function to query the locally loaded model and generate output.
    # Keep the existing assignment prompt template unchanged and only enforce
    # a strict 0/1 answer format via system instruction.

    # TODO(students): start
    messages = [
        {
            "role": "system",
            "content": "You are a sentiment classifier. Reply with only 1 for positive sentiment or 0 for negative sentiment.",
        },
        {"role": "user", "content": model_inputs},
    ]

    outputs = generator(
        messages,
        max_new_tokens=5,
        do_sample=False,
        return_full_text=False,
    )

    generated_text = outputs[0].get("generated_text", "")

    # Some transformers versions return chat messages, while others return a plain string.
    if isinstance(generated_text, list):
        last_item = generated_text[-1]
        if isinstance(last_item, dict):
            generated_text = last_item.get("content", "")
        else:
            generated_text = str(last_item)

    # TODO(students): end
    return str(generated_text).strip()


In [ ]:
def get_prompting_outputs(model_inputs):
    # input: model_inputs -> List
    # model_inputs is an array of the inputs to be fed to the LLM to obtain corresponding outputs (here, sentiment labels)
    # this function calls query(), which now uses local transformers inference
    predictions = []

    # TODO(students): start
    for model_input in tqdm(model_inputs):
        predictions.append(query(model_input))
    # TODO(students): end
    return predictions

### Download the gated model from Hugging Face (one-time setup)

This model is gated. Each student must:

1. Create/sign in to Hugging Face.
2. Request and accept access for `meta-llama/Llama-3.2-1B-Instruct`.
3. Create a HF token with read access.
4. In Colab, add the token in **Secrets** as `HF_TOKEN`, or enter it interactively (preferred).

How to add API token in **Secrets**

1. Open the Secrets tab: On the left-hand side of your Colab notebook, look for the '🔑' (key) icon. Click on it to open the Secrets panel.
2. Add a new secret: In the Secrets panel, click on the '+ Add new secret' button.
3. Enter details:
- For the 'Name' field, type HF_TOKEN (it's case-sensitive, so ensure it's exactly HF_TOKEN).
- For the 'Value' field, paste your Hugging Face token.
4. Save: Make sure the 'Notebook access' toggle is enabled for your current notebook. Then, close the Secrets panel.

Once added, your notebook can securely access this token using userdata.get('HF_TOKEN') without exposing it directly in your code.

**Safety Tip**: For security purposes, create a new token for this assignment and make sure you delete it after the course ends.


Run the next Python setup cell to authenticate and download the model to Drive (one-time setup). On later runs, it reuses local files.


After this step, the rest of the notebook runs using local files from Drive.

In [ ]:
from huggingface_hub import snapshot_download
import os

MODEL_REPO = "meta-llama/Llama-3.2-1B-Instruct"
LOCAL_MODEL_PATH = "/content/drive/MyDrive/models/Llama-3.2-1B-Instruct"

# One-time gated model download to Drive (reuses local files on later runs)
os.makedirs(LOCAL_MODEL_PATH, exist_ok=True)

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    from getpass import getpass
    hf_token = getpass("Enter your HF token (with access to meta-llama/Llama-3.2-1B-Instruct): ")

# Download only if model files are not already present
if not os.path.exists(os.path.join(LOCAL_MODEL_PATH, "config.json")):
    snapshot_download(
        repo_id=MODEL_REPO,
        local_dir=LOCAL_MODEL_PATH,
        token=hf_token,
        local_dir_use_symlinks=False,
    )

# Local inference pipeline
generator = pipeline(
    "text-generation",
    model=LOCAL_MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
)

In [ ]:
# Quick sanity check for local model inference
query("What is the capital of France?")

In [ ]:
# Not used in Path A (local inference).
API_URL = None
headers = {}

**Working Example**
--

First, we will walk through an example with just one data point and one prompt. Next, you will be required to experiment with designing different prompts on 100 data points, calculating the accuracy of sentiment classification using that prompt, and analyzing the outputs.

In [ ]:
test_set_slice_size = 1

In [ ]:
data_slice = data["test"].shuffle(seed=42).select([i for i in list(range(test_set_slice_size))])

In [ ]:
input_texts, ground_truth_labels = extract_input_outputs(data_slice)

Below is an example of how to write a prompt template, use it to generate input for the LLM and obtain outputs according to it. '{input_text}' is used as a placeholder for the input from the dataset. Make sure to retain it in your constructed prompt.

For example,
```
input_text = 'When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story'
prompt = 'review: {input_text}\n\nsentiment (1/0):'
```
renders as the following:
```
prompt = 'review: When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story

sentiment (1/0):'
```

In [ ]:
prompt = 'review: {input_text}\n\nsentiment (1/0):'
prompt_inputs = create_prompt_inputs(input_texts, prompt)
raw_generations = get_prompting_outputs(prompt_inputs)

In [ ]:
raw_generations

Model output looks similar to:
```
raw_generation = 'review: When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story

sentiment (1/0): 1'
```
We then need to extract just the model output so that we can transform it to 1/0 (integer) if needed so that we can evaluate accuracy of using the prompt.
If you notice the example above, the raw generation is the model input concatenated to the model output.
So, to extract just the output, we remove the model input from the raw generation and whatever is left is automatically the output.

In [ ]:
def extract_outputs_from_raw_generations(input_texts, raw_generations):
    model_outputs = []
    # TODO(students): start
    for input_text, raw_generation in zip(input_texts, raw_generations):
        # Older pipelines may return prompt + answer, while chat pipelines often
        # return only the assistant answer. Handle both cases safely.
        if raw_generation.startswith(input_text):
            model_outputs.append(raw_generation[len(input_text):].strip())
        else:
            model_outputs.append(raw_generation.strip())
    # TODO(students): end
    return model_outputs

In [ ]:
model_outputs = extract_outputs_from_raw_generations(prompt_inputs, raw_generations)

In [ ]:
print(f'Type of one item of model output: {type(model_outputs[0])}, one model output item: {model_outputs[0]}')

Above, you can see that our prompt design leads the LLM to generate a string which can be 0 or 1.

Now that we have the model outputs extracted, we now need to transform the outputs to 1/0 (integer) classes, so that we can compare the predicted labels to the ground truth labels and calculate accuracy when using that prompt.

In [ ]:
import re

def transform_demo_prompt_model_outputs_to_ground_truth_classes(model_outputs):
    predicted_labels = []

    # TODO(students): start
    for model_output in model_outputs:
        cleaned_output = model_output.strip()
        match = re.search(r"\b([01])\b", cleaned_output)

        # Use -1 for malformed generations so they count as incorrect.
        if match is None:
            predicted_labels.append(-1)
        else:
            predicted_labels.append(int(match.group(1)))
    # TODO(students): end
    return predicted_labels

In [ ]:
transformed_model_outputs = transform_demo_prompt_model_outputs_to_ground_truth_classes(model_outputs)

In [ ]:
print(f'Type of one item of model output after transformation: {type(transformed_model_outputs[0])}, one item of model output after transformation: {transformed_model_outputs[0]}')

In [ ]:
print(f'Type of one item of model output after transformation: {type(transformed_model_outputs[0])}, one item of model output after transformation: {transformed_model_outputs[0]}')

Above, you can see that the transformed model outputs are of type integer. These can now be successfully used to calculate accuracy of our method.

In [ ]:
accuracy = calculate_accuracy(transformed_model_outputs, ground_truth_labels)
print(accuracy)

**Prompt Design**
--

Tips on types of prompts to experiment with:
1. Instead of 1/0, you can prompt the model to output True/False, or positive/negative. Including information about the types of outputs you want nudges the model generation in the correct direction.
2. You can add an instruction at the start of the model input that the model should follow. It is good to have instructions that describe how you yourself would perform sentiment classification given a movie review.
3. LLMs have been shown to respond to certain types of artifacts as part of their prompts. For example, try adding "Let's think step by step" to your prompt and see whether performance of this LLM also improves. Note that the artifacts differ from model to model, and there is no guarantee that what works on one might also work on the other.
4. It also helps to prompt the models to reason about their answers in a structured manner. For example, you can prompt the model to generate a JSON object where one key denotes the value and another denotes the explanation for the answer.

In [ ]:
test_set_slice_size = 100

In [ ]:
data_slice = data["test"].shuffle(seed=42).select([i for i in list(range(test_set_slice_size))])

In [ ]:
input_texts, ground_truth_labels = extract_input_outputs(data_slice)

**Designing Prompt 1**
---

In [ ]:
prompt1 = 'review: {input_text}\n\nsentiment (1/0):'
prompt1_inputs = create_prompt_inputs(input_texts, prompt1)
raw_generations = get_prompting_outputs(prompt1_inputs)

**Optional Run**

For inspecting the entire results, you can run this cell.

In [ ]:
raw_generations

In [ ]:
print(raw_generations[0])

In [ ]:
model_outputs = extract_outputs_from_raw_generations(prompt1_inputs, raw_generations)

In [ ]:
print(f'Type of one item of model output: {type(model_outputs[0])}, one model output item: {model_outputs[0]}')

In [ ]:
def transform_prompt1_model_outputs_to_ground_truth_classes(model_outputs):
    # this is a custom function that works for the simple example we have been following till now
    # write a custom function to transform model outputs from your prompt to map to the ground truth classes
    # if model has hallucinated and generated something other than 0/1, it should also be treated as a failure case
    predicted_labels = []

    # TODO(students): start
    for model_output in model_outputs:
        cleaned_output = model_output.strip()
        match = re.search(r"\b([01])\b", cleaned_output)

        if match is None:
            predicted_labels.append(-1)
        else:
            predicted_labels.append(int(match.group(1)))
    # TODO(students): end
    return predicted_labels

In [ ]:
transformed_model_outputs = transform_demo_prompt_model_outputs_to_ground_truth_classes(model_outputs)

In [ ]:
print(f'Type of one item of model output after transformation: {type(transformed_model_outputs[0])}, one item of model output after transformation: {transformed_model_outputs[0]}')

In [ ]:
accuracy = calculate_accuracy(transformed_model_outputs, ground_truth_labels)
print(f'Accuracy of sentiment classification by using this prompt: {accuracy}%')

In [ ]:
df = pd.DataFrame({'review': input_texts, 'ground_truth_label': ground_truth_labels, 'model_input': prompt1_inputs, 'raw_model_output': model_outputs, 'transformed_model_output': transformed_model_outputs})
df.to_csv('cse354_assignment4_submission_prompt1_outputs.csv', index=False)


---
# **Part 2: RewardBench Subset Analysis — Reward Scores and Reward Artifacts**

In this section, you will use the same Llama Instruct model as a reward model. The goal is **not** to build a perfect reward model. The goal is to investigate whether a simple LLM-based reward scoring mechanism relies on superficial artifacts such as response length, politeness markers, refusal language, or formatting.

RewardBench contains single-turn preference examples with a `prompt`, a preferred answer (`chosen`), a less preferred answer (`rejected`), and a `subset` label. In standard RewardBench evaluation, a reward model succeeds on an example when it assigns a higher score to the `chosen` response than to the `rejected` response.

## Required tasks

**Task 1: Reward scoring mechanism**

Create a reward scoring mechanism using the Llama Instruct model. For each `(prompt, response)` pair, your scoring function should return a numeric reward score. The provided starter function asks Llama to judge the response on a 1--5 scale and extracts the numeric score. You may improve the prompt, parsing, or scoring method, but you must clearly explain your design choice.

**Task 2: Artifact correlation analysis**

After computing reward scores, define several response artifacts and test whether the reward score correlates with them. You must include response length and at least two more artifacts. Examples include:

- response length, measured by number of words or characters;
- refusal language, e.g., phrases such as "I cannot", "I can't", "I’m unable", "as an AI";
- politeness markers, e.g., "please", "thank you", "sorry", "happy to help";
- formatting artifacts, e.g., bullet points, numbered lists, markdown headings;
- hedging / uncertainty markers, e.g., "maybe", "might", "possibly".

For each artifact, plot artifact presence or magnitude versus reward score and compute a correlation measure such as Pearson or Spearman correlation. Then write a short interpretation: does the reward model appear to reward quality, or is it partially relying on shortcuts?


In [ ]:

# RewardBench setup
# This cell downloads a small subset of RewardBench and converts chosen/rejected responses
# into a flat table of prompt-response pairs.

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm import tqdm

# Keep this small so the assignment runs on Colab. Increase after testing if desired.
REWARDBENCH_SAMPLE_SIZE = 20
REWARDBENCH_SEED = 42

# Good starter subsets for artifact analysis:
# - "alpacaeval-easy": strong length artifact; chosen answers are often much longer.
# - "alpacaeval-length": controls length better, so superficial length should matter less.
# - "refusals-dangerous" or "refusals-offensive": useful for studying refusal markers.
REWARDBENCH_SUBSET = "alpacaeval-easy"


def stringify_rewardbench_field(value):
    """Convert RewardBench fields to plain strings, even if a subset stores chat messages."""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                content = item.get("content", "")
                role = item.get("role", "")
                parts.append(f"{role}: {content}".strip(": "))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    if isinstance(value, dict):
        return value.get("content", str(value))
    return str(value)


# The RewardBench dataset provides preference pairs with prompt/chosen/rejected fields.
try:
    rewardbench = load_dataset("allenai/reward-bench", split="filtered")
except ValueError:
    rewardbench = load_dataset("allenai/reward-bench", split="train")

available_subsets = sorted(set(rewardbench["subset"]))
print(f"Loaded RewardBench with {len(rewardbench)} examples.")
print("First 10 available subsets:", available_subsets[:10])

subset_pool = rewardbench.filter(lambda ex: ex["subset"] == REWARDBENCH_SUBSET)

if len(subset_pool) == 0:
    print(f"Warning: subset '{REWARDBENCH_SUBSET}' was not found. Using the first available examples instead.")
    subset_pool = rewardbench

sample_n = min(REWARDBENCH_SAMPLE_SIZE, len(subset_pool))
rewardbench_subset = subset_pool.shuffle(seed=REWARDBENCH_SEED).select(range(sample_n))

rows = []
for row_id, ex in enumerate(rewardbench_subset):
    ex_id = ex.get("id", f"rb_{row_id}")
    prompt_text = stringify_rewardbench_field(ex["prompt"])
    chosen_text = stringify_rewardbench_field(ex["chosen"])
    rejected_text = stringify_rewardbench_field(ex["rejected"])

    rows.append({
        "id": ex_id,
        "subset": ex.get("subset", REWARDBENCH_SUBSET),
        "prompt": prompt_text,
        "response": chosen_text,
        "label": "chosen",
        "is_chosen": 1,
    })
    rows.append({
        "id": ex_id,
        "subset": ex.get("subset", REWARDBENCH_SUBSET),
        "prompt": prompt_text,
        "response": rejected_text,
        "label": "rejected",
        "is_chosen": 0,
    })

rb_df = pd.DataFrame(rows)
print(rb_df.shape)
rb_df.head()



# Task 1: use Llama Instruct as a weak reward model.

#### You should come up with their own rewarding scheme or mechanism. These two rewarding schemes are given as reference for you to gain knowledge.

#Task 1: Reward model 1: Example 1
#### The model is asked to assign a numeric score from 1 to 5.

In [ ]:


def build_reward_prompt(instruction, response):
    return f"""You are evaluating the quality of an assistant response.

Instruction:
{instruction}

Assistant response:
{response}

Give a reward score from 1 to 5, where:
1 = very poor, unsafe, irrelevant, or unhelpful
2 = weak
3 = acceptable
4 = good
5 = excellent, helpful, safe, and well-aligned

Reply with only one decimal number from 1 to 5 only.
Reward score:"""


def parse_reward_score(text):
    """Extract the first score in (1, 5). Return np.nan if parsing fails."""
    match = re.search(r"\b(10(?:\.0+)?|[0-5](?:\.\d+)?)\b", text)
    if match:
        return float(match.group(1))
    return np.nan


def extract_chat_pipeline_text(output):
    """Handle common transformers text-generation pipeline return formats."""
    generated = output[0].get("generated_text", "")
    if isinstance(generated, list):
        last_message = generated[-1]
        if isinstance(last_message, dict):
            return str(last_message.get("content", "")).strip()
        return str(last_message).strip()
    return str(generated).strip()


def reward_query(instruction, response):
    """Query the already-loaded Llama Instruct generation pipeline for a reward score."""
    reward_prompt = build_reward_prompt(instruction, response)
    messages = [
        {"role": "system", "content": "You are a strict reward model. Reply with exactly one decimal number from 1 to 5 for the relavence of Assistance response to the instruction."},
        {"role": "user", "content": reward_prompt},
    ]
    output = generator(
        messages,
        max_new_tokens=3,
        do_sample=False,
        return_full_text=False,
    )
    return extract_chat_pipeline_text(output)


# Compute rewards. This may take a few minutes depending on hardware.
raw_reward_outputs = []
reward_scores = []

for _, row in tqdm(rb_df.iterrows(), total=len(rb_df)):
    raw_score_text = reward_query(row["prompt"], row["response"])
    raw_reward_outputs.append(raw_score_text)
    reward_scores.append(parse_reward_score(raw_score_text))

rb_df["raw_reward_output"] = raw_reward_outputs
rb_df["reward_score"] = reward_scores

# If a few generations fail to parse, keep the rows but report the issue.
num_unparsed = rb_df["reward_score"].isna().sum()
print(f"Unparsed reward scores: {num_unparsed} / {len(rb_df)}")

rb_df[["id", "label", "raw_reward_output", "reward_score", "response"]].head()


#Task 1: Reward model 2: Example 2

In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm
from scipy.special import expit

# Get model/tokenizer directly from the pipeline
model = generator.model
tokenizer = generator.tokenizer
model.eval()

def build_reward_prompt(instruction, response):
    return f"""You are judging whether an assistant response is good.

User instruction:
{instruction}

Assistant response:
{response}

Is this response helpful, correct, relevant, and safe?

Answer only Yes or No.
Answer:"""

def reward_score_from_logits(instruction, response):
    prompt = build_reward_prompt(instruction, response)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # logits for the next token after "Answer:"
    next_token_logits = outputs.logits[0, -1, :]

    yes_ids = tokenizer.encode(" Yes", add_special_tokens=False)
    no_ids = tokenizer.encode(" No", add_special_tokens=False)

    yes_id = yes_ids[0]
    no_id = no_ids[0]

    yes_logit = next_token_logits[yes_id].item()
    no_logit = next_token_logits[no_id].item()

    # probability that the answer is Yes
    prob_yes = expit(yes_logit - no_logit)

    # convert probability to a 1–5 reward score
    reward_score = 1.0 + 4.0 * prob_yes

    return reward_score, prob_yes

reward_scores = []
prob_yes_values = []

for _, row in tqdm(rb_df.iterrows(), total=len(rb_df)):
    score, prob_yes = reward_score_from_logits(row["prompt"], row["response"])
    reward_scores.append(score)
    prob_yes_values.append(prob_yes)

rb_df["reward_score"] = reward_scores
rb_df["prob_yes"] = prob_yes_values

rb_df[["id", "label", "reward_score", "prob_yes", "response"]].head()

##**Your Task is to design a completely new reward model.**
###Create a reward scoring mechanism using the Llama Instruct model. For each (prompt, response) pair, your scoring function should return a numeric reward score. The provided starter function asks Llama to judge the response on a 1--5 scale and extracts the numeric score. Explain your design process.bold text

In [ ]:
###TODO###

#Your reward model

###END TODO##

# RewardBench-style pairwise evaluation.

In [ ]:

# A reward model is correct if reward(chosen) > reward(rejected) for the same prompt.

pair_scores = rb_df.pivot_table(index="id", columns="label", values="reward_score", aggfunc="first").reset_index()

for required_col in ["chosen", "rejected"]:
    if required_col not in pair_scores.columns:
        pair_scores[required_col] = np.nan

valid_pairs = pair_scores.dropna(subset=["chosen", "rejected"]).copy()
valid_pairs["reward_model_correct"] = valid_pairs["chosen"] > valid_pairs["rejected"]
valid_pairs["tie"] = valid_pairs["chosen"] == valid_pairs["rejected"]

pairwise_accuracy = valid_pairs["reward_model_correct"].mean() if len(valid_pairs) > 0 else np.nan
tie_rate = valid_pairs["tie"].mean() if len(valid_pairs) > 0 else np.nan

print(f"Pairwise RewardBench accuracy on {REWARDBENCH_SUBSET}: {pairwise_accuracy:.3f}")
print(f"Tie rate: {tie_rate:.3f}")

valid_pairs.head()


#Task 2


In [ ]:

# Task 2 starter: you should define at least 2 additional artifacts.
# You must include word count(length) and at least two additional artifacts.

def count_matches(text, patterns):
    text = str(text).lower()
    return sum(len(re.findall(pattern, text)) for pattern in patterns)

#example artifact
REFUSAL_PATTERNS = [
    r"\bi cannot\b", r"\bi can't\b", r"\bi am unable\b", r"\bi'm unable\b",
    r"\bi cannot assist\b", r"\bi can't help\b", r"\bas an ai\b", r"\bi won'?t\b",
]

##TODO##

#Define artifacts

##END TODO##

rb_df["refusal_count"] = rb_df["response"].apply(lambda x: count_matches(x, REFUSAL_PATTERNS))
rb_df["has_refusal"] = (rb_df["refusal_count"] > 0).astype(int)

##TODO##

##END TODO##

artifact_columns = [
    "has_refusal",
    "refusal_count",
    ##TODO##

    ##END TODO##
]

rb_df[["label", "reward_score"] + artifact_columns].head()


### Correlation table.
#### Pearson captures linear association.
#### Spearman captures monotonic rank association and is computed using ranks to avoid extra dependencies.

In [ ]:
correlation_rows = []
for col in artifact_columns:
    temp = rb_df[[col, "reward_score"]].dropna()
    if temp[col].nunique() <= 1 or temp["reward_score"].nunique() <= 1:
        pearson = np.nan
        spearman = np.nan
    else:
        pearson = temp[col].corr(temp["reward_score"], method="pearson")
        spearman = temp[col].rank().corr(temp["reward_score"].rank(), method="pearson")
    correlation_rows.append({
        "artifact": col,
        "pearson_corr_with_reward": pearson,
        "spearman_corr_with_reward": spearman,
    })

corr_df = pd.DataFrame(correlation_rows)
corr_df["abs_spearman"] = corr_df["spearman_corr_with_reward"].abs()
corr_df = corr_df.sort_values(by="abs_spearman", ascending=False).drop(columns=["abs_spearman"])

corr_df


###Plots

In [ ]:

# Plot artifact versus reward score.
# You should plot the artifacts vs. reward Score for each type of artifact

##TODO


##END TODO##


### Task: Reward Delta vs Artifact Delta Analysis
#### Your task is to
#### Plot:
####   predicted_reward_delta = RM(chosen) - RM(rejected)
#### against:
####   artifact_delta = artifact(chosen) - artifact(rejected)
#### for each prompt.
#### Then compute Pearson correlation.
#####RM :  Reward Model


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Build pairwise dataframe from rb_df

pair_rows = []

for ex_id, group in rb_df.groupby("id"):

    chosen_rows = group[group["label"] == "chosen"]
    rejected_rows = group[group["label"] == "rejected"]

    if len(chosen_rows) == 0 or len(rejected_rows) == 0:
        continue

    chosen = chosen_rows.iloc[0]
    rejected = rejected_rows.iloc[0]

    pair_rows.append({
        "id": ex_id,

        "chosen_response": chosen["response"],
        "rejected_response": rejected["response"],

        "chosen_reward": chosen["reward_score"],
        "rejected_reward": rejected["reward_score"],
    })

pair_df = pd.DataFrame(pair_rows)

pair_df["reward_delta"] = (
    pair_df["chosen_reward"] - pair_df["rejected_reward"]
)


#TODO

# compute artifact deltas for 2 artifacts of your choice including word count artifact.

#END TODO

#TODO

# Pearson correlation Reward Delta vs artifact Delta for each artifact

#TODO


##TODO

#plot Reward Delta vs Word Count Delta for each of selected two artifacts

##END TODO



## Student written response

Answer the following questions in 1--2 paragraphs each.

1. **Reward scoring mechanism:** What prompt or method did you use to convert Llama Instruct into a reward model? Why is this a reasonable but imperfect reward mechanism?
2. **RewardBench performance:** What pairwise accuracy did your reward model achieve on your selected RewardBench subset? Did it often tie chosen and rejected responses?
3. **Artifact correlations:** Which artifacts had the strongest correlation with reward score? Include at least one plot and one correlation measure in your explanation.
4. **Interpretation:** Based on your results, does the reward model appear to use superficial heuristics? For example, does it reward longer answers, refusals, politeness, or formatting even when those features may not imply response quality?
5. **Extension:** Repeat the analysis on a different subset, such as `alpacaeval-length`, `refusals-dangerous`, or `xstest-should-respond`. Do the same artifacts still correlate with reward score?


In [ ]:
###TODO

##Answers to the questions

###END TODO

## **Submission guidelines**
---
You would need to submit the following files:

1.   `NLP_HW3.ipynb` - This ipynb file. It will also work as your report, so please add a description to your code and use text cells for providing written answers wherever required.
2.   `gdrive_link.txt` - Should contain a wgetable to your Base Directory (as instructed in **Mounting your drive**).

## Collaboration Guidelines

  - You can collaborate to discuss ideas and to help each other for better understanding of concepts and math.
  - You should NOT collaborate on the code level. This includes all implementation activities: design, coding, and debugging.
  - You should NOT not use any code that you did not write to complete the assignment.
  - The homework will be **cross-checked**. Do not cheat at all! It’s worth doing the homework partially instead of cheating and copying your code and get 0 for the whole homework. In previous years, students have faced harsh disciplinary action as a result of the same.
